# NEXUS-AURORA: Evaluation

Loads a trained checkpoint and runs:
1. BPB (bits per byte) evaluation on validation set
2. Routing analysis (K distribution, surprise statistics)
3. Sample generation
4. Comparison table: AURORA vs LLaMA baseline

In [ ]:
import sys
sys.path.insert(0, '/kaggle/working/nexus-lm')

import torch
import yaml

CKPT_PATH  = '/kaggle/working/checkpoints/ckpt_tokens_1500M.pt'
VAL_BIN    = '/kaggle/input/nexus-data/fineweb_edu_val.bin'
CONFIG     = '/kaggle/working/nexus-lm/config/nexus_aurora_v1.yaml'
TOKENIZER  = '/kaggle/input/nexus-data/tokenizer.model'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# Load model
from model.model import NexusAurora, AuroraConfig

with open(CONFIG) as f:
    cfg = yaml.safe_load(f)

model = NexusAurora(AuroraConfig(**cfg['model']))
ckpt = torch.load(CKPT_PATH, map_location=device)
model.load_state_dict(ckpt['model_state'])
model = model.to(device).eval()
print(f'Loaded checkpoint from step {ckpt["step"]} ({ckpt["tokens_seen"]:,} tokens)')

In [ ]:
# BPB evaluation
from data.dataloader import MemmapDataset, get_dataloader
from evaluation.perplexity import compute_bpb

val_ds     = MemmapDataset(VAL_BIN, seq_len=cfg['model']['max_seq_len'])
val_loader = get_dataloader(val_ds, batch_size=16, shuffle=False)
bpb = compute_bpb(model, val_loader, device, torch.float16)
print(f'AURORA BPB: {bpb:.4f}  (target: ≤ 0.94, GPT-2 ~1.10)')

In [ ]:
# Routing analysis
from evaluation.routing_analysis import analyze_routing

routing = analyze_routing(model, val_loader, device, n_batches=50)
print('\nRouting Analysis:')
print(f'  K=1 fraction: {routing["k_1"]:.3f}')
print(f'  K=2 fraction: {routing["k_2"]:.3f}')
print(f'  K=4 fraction: {routing["k_4"]:.3f}')
print(f'  Surprise mean: {routing["surprise_mean"]:.4f}')
print(f'  Surprise std:  {routing["surprise_std"]:.4f}')

In [ ]:
# Sample generation
from data.tokenizer import load_tokenizer, encode, decode

sp = load_tokenizer(TOKENIZER)
prompt = 'The theory of relativity states that'
ids = torch.tensor([encode(sp, prompt)], device=device)

with torch.no_grad():
    generated = model.generate(ids, max_new_tokens=100, temperature=0.8, top_k=50)

print('Prompt:', prompt)
print('Generated:', decode(sp, generated[0].tolist()[len(ids[0]):]))

In [ ]:
# Visualize K-routing distribution
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# K distribution pie chart
k_vals = [routing['k_1'], routing['k_2'], routing['k_4']]
axes[0].pie(k_vals, labels=['K=1', 'K=2', 'K=4'],
            autopct='%1.1f%%', colors=['#3498db', '#2ecc71', '#e74c3c'])
axes[0].set_title('Reasoning Iteration (K) Distribution')

# Surprise histogram (approximate)
axes[1].bar(['Mean Surprise', 'Std Surprise'],
            [routing['surprise_mean'], routing['surprise_std']],
            color=['#9b59b6', '#1abc9c'])
axes[1].set_ylabel('Value')
axes[1].set_title('Verification Stream Statistics')

plt.tight_layout()
plt.savefig('/kaggle/working/routing_analysis.png', dpi=150)
plt.show()
print(f'BPB: {bpb:.4f}')